In [16]:
import requests
import pandas as pd
from datetime import datetime
import logging
from typing import Optional
import json

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

class KOSISRetailSalesCollector:
    """KOSIS 제품 판매채널별 월별 판매액 데이터 수집"""

    def __init__(self, api_key: str):
        self.api_key = api_key
        self.base_url = "https://kosis.kr/openapi/Param/statisticsParameterData.do"

    def get_retail_sales_data(self, months: int = 24) -> Optional[pd.DataFrame]:
        """
        판매채널별 판매액 데이터 수집

        Parameters:
        -----------
        months : int
            최근 몇 개월 데이터 (기본값: 24개월)
        """
        params = {
            'method': 'getList',
            'apiKey': self.api_key,
            'itmId': 'T1',  # 판매액
            'objL1': 'ALL',  # 전체 업태
            'objL2': '',
            'objL3': '',
            'objL4': '',
            'objL5': '',
            'objL6': '',
            'objL7': '',
            'objL8': '',
            'format': 'json',
            'jsonVD': 'Y',
            'prdSe': 'M',  # 월별
            'newEstPrdCnt': str(months),
            'orgId': '101',
            'tblId': 'DT_1K41003'
        }

        try:
            logger.info(f"KOSIS 판매채널별 판매액 데이터 요청 (최근 {months}개월)")
            response = requests.get(self.base_url, params=params, timeout=30)
            response.raise_for_status()

            data = response.json()

            if not data:
                logger.warning("응답 데이터가 비어있습니다")
                return None

            df = pd.DataFrame(data)
            logger.info(f"원본 데이터 수집 완료: {len(df)} rows")
            logger.info(f"컬럼: {df.columns.tolist()}")

            return df

        except requests.exceptions.RequestException as e:
            logger.error(f"API 요청 실패: {e}")
            return None
        except Exception as e:
            logger.error(f"데이터 처리 중 오류: {e}")
            return None

    def process_data(self, df: pd.DataFrame) -> pd.DataFrame:
        """데이터 전처리 및 정제"""
        if df is None or df.empty:
            logger.warning("처리할 데이터가 없습니다")
            return pd.DataFrame()

        logger.info("데이터 전처리 시작")
        processed = df.copy()

        # 1. 날짜 처리
        if 'PRD_DE' in processed.columns:
            processed['date'] = pd.to_datetime(
                processed['PRD_DE'],
                format='%Y%m',
                errors='coerce'
            )
            processed['year'] = processed['date'].dt.year
            processed['month'] = processed['date'].dt.month
            processed['year_month'] = processed['PRD_DE']

        # 2. 판매채널 정보
        channel_cols = ['C1_NM', 'C2_NM', 'C1', 'C2']
        for col in channel_cols:
            if col in processed.columns:
                processed[f'channel_{col.lower()}'] = processed[col]

        # 3. 판매액 (숫자로 변환)
        if 'DT' in processed.columns:
            processed['sales_amount'] = pd.to_numeric(
                processed['DT'].astype(str).str.replace(',', '').str.strip(),
                errors='coerce'
            )

        # 4. 단위 정보
        if 'UNIT_NM' in processed.columns:
            processed['unit'] = processed['UNIT_NM']

        # 5. 항목명
        if 'ITM_NM' in processed.columns:
            processed['item_name'] = processed['ITM_NM']

        # 6. 통계표 정보
        if 'TBL_NM' in processed.columns:
            processed['table_name'] = processed['TBL_NM']

        if 'TBL_ID' in processed.columns:
            processed['table_id'] = processed['TBL_ID']

        if 'ORG_ID' in processed.columns:
            processed['org_id'] = processed['ORG_ID']

        # 7. 데이터 수집 시간
        processed['collected_at'] = datetime.now()

        # 8. 결측치 확인
        logger.info(f"결측치 확인:\n{processed.isnull().sum()}")

        return processed

    def create_summary(self, df: pd.DataFrame) -> pd.DataFrame:
        """판매채널별 요약 통계"""
        if df.empty or 'sales_amount' not in df.columns:
            logger.warning("요약 통계를 생성할 수 없습니다")
            return pd.DataFrame()

        # 채널 컬럼 찾기
        channel_col = None
        for col in ['channel_c1_nm', 'channel_c2_nm', 'C1_NM', 'C2_NM']:
            if col in df.columns:
                channel_col = col
                break

        if channel_col is None:
            logger.warning("채널 정보를 찾을 수 없습니다")
            return pd.DataFrame()

        summary = df.groupby(channel_col).agg({
            'sales_amount': ['count', 'mean', 'sum', 'min', 'max', 'std'],
            'date': ['min', 'max']
        }).round(2)

        summary.columns = ['_'.join(col).strip() for col in summary.columns.values]
        summary = summary.reset_index()
        summary.columns = [
            'channel', 'record_count', 'avg_sales', 'total_sales',
            'min_sales', 'max_sales', 'std_sales', 'first_date', 'last_date'
        ]

        summary = summary.sort_values('total_sales', ascending=False)

        return summary

    def create_pivot_table(self, df: pd.DataFrame) -> pd.DataFrame:
        """날짜 x 판매채널 피벗 테이블 생성"""
        if df.empty:
            return pd.DataFrame()

        # 채널 컬럼 찾기
        channel_col = None
        for col in ['channel_c1_nm', 'channel_c2_nm', 'C1_NM', 'C2_NM']:
            if col in df.columns:
                channel_col = col
                break

        if channel_col is None or 'date' not in df.columns:
            logger.warning("피벗 테이블을 생성할 수 없습니다")
            return pd.DataFrame()

        pivot = df.pivot_table(
            values='sales_amount',
            index='date',
            columns=channel_col,
            aggfunc='sum'
        )

        pivot = pivot.sort_index()

        return pivot

    def get_channel_timeseries(self, df: pd.DataFrame, channel: str = None) -> pd.DataFrame:
        """특정 채널의 시계열 데이터"""
        if df.empty:
            return pd.DataFrame()

        # 채널 컬럼 찾기
        channel_col = None
        for col in ['channel_c1_nm', 'channel_c2_nm', 'C1_NM', 'C2_NM']:
            if col in df.columns:
                channel_col = col
                break

        if channel_col is None:
            return df[['date', 'sales_amount']].sort_values('date')

        if channel:
            filtered = df[df[channel_col] == channel].copy()
            filtered = filtered[['date', channel_col, 'sales_amount', 'unit']].sort_values('date')
            return filtered
        else:
            return df[['date', channel_col, 'sales_amount', 'unit']].sort_values('date')

    def calculate_growth_rate(self, df: pd.DataFrame) -> pd.DataFrame:
        """전년 동월 대비 성장률 계산"""
        if df.empty or 'date' not in df.columns:
            return pd.DataFrame()

        # 채널 컬럼 찾기
        channel_col = None
        for col in ['channel_c1_nm', 'channel_c2_nm', 'C1_NM', 'C2_NM']:
            if col in df.columns:
                channel_col = col
                break

        if channel_col is None:
            return pd.DataFrame()

        df_sorted = df.sort_values(['date', channel_col]).copy()

        # 전년 동월 대비 성장률
        df_sorted['yoy_growth'] = df_sorted.groupby(channel_col)['sales_amount'].pct_change(12) * 100

        # 전월 대비 성장률
        df_sorted['mom_growth'] = df_sorted.groupby(channel_col)['sales_amount'].pct_change(1) * 100

        return df_sorted

    def create_yoy_growth_pivot(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        전년 동월 대비 성장률(YoY Growth) 피벗 테이블 생성
        날짜 x 판매채널, 값은 YoY 성장률(%)

        Parameters:
        -----------
        df : pd.DataFrame
            calculate_growth_rate()로 성장률이 계산된 데이터프레임

        Returns:
        --------
        pd.DataFrame
            날짜를 인덱스로, 채널을 컬럼으로 하는 YoY 성장률 피벗 테이블
        """
        if df.empty:
            logger.warning("데이터가 비어있습니다")
            return pd.DataFrame()

        if 'yoy_growth' not in df.columns:
            logger.warning("yoy_growth 컬럼이 없습니다. calculate_growth_rate()를 먼저 실행하세요")
            return pd.DataFrame()

        # 채널 컬럼 찾기
        channel_col = None
        for col in ['channel_c1_nm', 'channel_c2_nm', 'C1_NM', 'C2_NM']:
            if col in df.columns:
                channel_col = col
                break

        if channel_col is None or 'date' not in df.columns:
            logger.warning("날짜 또는 채널 정보를 찾을 수 없습니다")
            return pd.DataFrame()

        # YoY 성장률 피벗 테이블 생성
        yoy_pivot = df.pivot_table(
            values='yoy_growth',
            index='date',
            columns=channel_col,
            aggfunc='mean'  # 중복이 있을 경우 평균 사용
        )

        yoy_pivot = yoy_pivot.sort_index()

        # 소수점 2자리로 반올림
        yoy_pivot = yoy_pivot.round(2)

        logger.info(f"YoY 성장률 피벗 테이블 생성 완료: {yoy_pivot.shape}")

        return yoy_pivot

    def create_mom_growth_pivot(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        전월 대비 성장률(MoM Growth) 피벗 테이블 생성
        날짜 x 판매채널, 값은 MoM 성장률(%)

        Parameters:
        -----------
        df : pd.DataFrame
            calculate_growth_rate()로 성장률이 계산된 데이터프레임

        Returns:
        --------
        pd.DataFrame
            날짜를 인덱스로, 채널을 컬럼으로 하는 MoM 성장률 피벗 테이블
        """
        if df.empty:
            logger.warning("데이터가 비어있습니다")
            return pd.DataFrame()

        if 'mom_growth' not in df.columns:
            logger.warning("mom_growth 컬럼이 없습니다. calculate_growth_rate()를 먼저 실행하세요")
            return pd.DataFrame()

        # 채널 컬럼 찾기
        channel_col = None
        for col in ['channel_c1_nm', 'channel_c2_nm', 'C1_NM', 'C2_NM']:
            if col in df.columns:
                channel_col = col
                break

        if channel_col is None or 'date' not in df.columns:
            logger.warning("날짜 또는 채널 정보를 찾을 수 없습니다")
            return pd.DataFrame()

        # MoM 성장률 피벗 테이블 생성
        mom_pivot = df.pivot_table(
            values='mom_growth',
            index='date',
            columns=channel_col,
            aggfunc='mean'
        )

        mom_pivot = mom_pivot.sort_index()

        # 소수점 2자리로 반올림
        mom_pivot = mom_pivot.round(2)

        logger.info(f"MoM 성장률 피벗 테이블 생성 완료: {mom_pivot.shape}")

        return mom_pivot

    def export_to_excel(self, df: pd.DataFrame, summary: pd.DataFrame,
                       pivot: pd.DataFrame, yoy_pivot: pd.DataFrame = None,
                       mom_pivot: pd.DataFrame = None,
                       filename: str = 'kosis_retail_sales.xlsx'):
        """여러 시트로 구성된 Excel 파일로 저장"""
        try:
            with pd.ExcelWriter(filename, engine='openpyxl') as writer:
                if not df.empty:
                    df.to_excel(writer, sheet_name='원본데이터', index=False)

                if not summary.empty:
                    summary.to_excel(writer, sheet_name='채널별요약', index=False)

                if not pivot.empty:
                    pivot.to_excel(writer, sheet_name='판매액_피벗')

                # 성장률 데이터
                growth_df = self.calculate_growth_rate(df)
                if not growth_df.empty:
                    growth_df.to_excel(writer, sheet_name='성장률분석', index=False)

                # YoY 성장률 피벗 테이블
                if yoy_pivot is not None and not yoy_pivot.empty:
                    yoy_pivot.to_excel(writer, sheet_name='YoY성장률_피벗')

                # MoM 성장률 피벗 테이블
                if mom_pivot is not None and not mom_pivot.empty:
                    mom_pivot.to_excel(writer, sheet_name='MoM성장률_피벗')

            logger.info(f"Excel 파일 저장 완료: {filename}")

        except Exception as e:
            logger.error(f"Excel 저장 실패: {e}")

    def run_collection(self, months: int = 24, save_excel: bool = True) -> dict:
        """
        전체 수집 프로세스 실행

        Returns:
        --------
        dict: 'raw', 'processed', 'summary', 'pivot', 'yoy_pivot', 'mom_pivot' 키를 가진 딕셔너리
        """
        logger.info("=== KOSIS 판매채널별 판매액 데이터 수집 시작 ===")

        # 1. 데이터 수집
        raw_data = self.get_retail_sales_data(months)
        if raw_data is None or raw_data.empty:
            logger.error("데이터 수집 실패")
            return {
                'raw': pd.DataFrame(),
                'processed': pd.DataFrame(),
                'summary': pd.DataFrame(),
                'pivot': pd.DataFrame(),
                'growth': pd.DataFrame(),
                'yoy_pivot': pd.DataFrame(),
                'mom_pivot': pd.DataFrame()
            }

        # 2. 데이터 처리
        processed_data = self.process_data(raw_data)

        # 3. 요약 통계 생성
        summary = self.create_summary(processed_data)
        if not summary.empty:
            logger.info("\n=== 판매채널별 요약 통계 ===")
            print(summary.to_string(index=False))

        # 4. 판매액 피벗 테이블 생성
        pivot = self.create_pivot_table(processed_data)
        if not pivot.empty:
            logger.info(f"\n=== 판매액 피벗 테이블 (날짜 x 채널) ===")
            print(pivot.head(10))

        # 5. 성장률 계산
        growth_data = self.calculate_growth_rate(processed_data)

        # 6. YoY 성장률 피벗 테이블 생성
        yoy_pivot = self.create_yoy_growth_pivot(growth_data)
        if not yoy_pivot.empty:
            logger.info(f"\n=== YoY 성장률 피벗 테이블 (날짜 x 채널, 단위: %) ===")
            print(yoy_pivot.tail(12))  # 최근 12개월

        # 7. MoM 성장률 피벗 테이블 생성
        mom_pivot = self.create_mom_growth_pivot(growth_data)
        if not mom_pivot.empty:
            logger.info(f"\n=== MoM 성장률 피벗 테이블 (날짜 x 채널, 단위: %) ===")
            print(mom_pivot.tail(12))  # 최근 12개월

        # 8. Excel 저장
        if save_excel:
            self.export_to_excel(processed_data, summary, pivot, yoy_pivot, mom_pivot)

        logger.info("=== 데이터 수집 완료 ===")

        return {
            'raw': raw_data,
            'processed': processed_data,
            'summary': summary,
            'pivot': pivot,
            'growth': growth_data,
            'yoy_pivot': yoy_pivot,
            'mom_pivot': mom_pivot
        }


def main():
    # API 키 설정
    kosis_key = "ZTFhMjg1MzhmNmFiYWJlYmY3ZWUxZDA0ZDI2ZTM0YWU="

    # 수집기 초기화
    collector = KOSISRetailSalesCollector(kosis_key)

    # 데이터 수집 실행 (최근 36개월 - YoY 계산을 위해 충분한 기간 필요)
    results = collector.run_collection(months=36, save_excel=True)

    # 결과 활용 예시
    processed_df = results['processed']
    summary_df = results['summary']
    pivot_df = results['pivot']
    yoy_pivot_df = results['yoy_pivot']
    mom_pivot_df = results['mom_pivot']

    # YoY 성장률 피벗 테이블 상세 분석
    if not yoy_pivot_df.empty:
        print("\n=== YoY 성장률 피벗 테이블 전체 ===")
        print(yoy_pivot_df)

        print("\n=== 최근 6개월 YoY 성장률 ===")
        print(yoy_pivot_df.tail(6))

        print("\n=== 채널별 평균 YoY 성장률 ===")
        print(yoy_pivot_df.mean().sort_values(ascending=False))

        print("\n=== 채널별 최근 YoY 성장률 (가장 최근 월) ===")
        print(yoy_pivot_df.iloc[-1].sort_values(ascending=False))


if __name__ == "__main__":
    main()

2026-02-11 21:26:20,680 - INFO - === KOSIS 판매채널별 판매액 데이터 수집 시작 ===
2026-02-11 21:26:20,681 - INFO - KOSIS 판매채널별 판매액 데이터 요청 (최근 36개월)
2026-02-11 21:26:20,948 - INFO - 원본 데이터 수집 완료: 288 rows
2026-02-11 21:26:20,948 - INFO - 컬럼: ['C1_OBJ_NM', 'DT', 'C1', 'PRD_SE', 'UNIT_NM_ENG', 'ITM_ID', 'TBL_ID', 'ITM_NM', 'TBL_NM', 'PRD_DE', 'LST_CHN_DE', 'C1_NM_ENG', 'C1_NM', 'UNIT_NM', 'ITM_NM_ENG', 'ORG_ID', 'C1_OBJ_NM_ENG']
2026-02-11 21:26:20,950 - INFO - 데이터 전처리 시작
2026-02-11 21:26:20,956 - INFO - 결측치 확인:
C1_OBJ_NM        0
DT               0
C1               0
PRD_SE           0
UNIT_NM_ENG      0
ITM_ID           0
TBL_ID           0
ITM_NM           0
TBL_NM           0
PRD_DE           0
LST_CHN_DE       0
C1_NM_ENG        0
C1_NM            0
UNIT_NM          0
ITM_NM_ENG       0
ORG_ID           0
C1_OBJ_NM_ENG    0
date             0
year             0
month            0
year_month       0
channel_c1_nm    0
channel_c1       0
sales_amount     0
unit             0
item_name        0
table_

     channel  record_count   avg_sales  total_sales  min_sales  max_sales  std_sales first_date  last_date
       전문소매점            36 15691888.94    564908002   14491139   17026074  677373.14 2023-01-01 2025-12-01
      무점포 소매            36 11436106.17    411699822    9986914   12709635  603605.65 2023-01-01 2025-12-01
승용차 및 연료 소매점            36 10824028.75    389665035    9274173   12266219  684763.12 2023-01-01 2025-12-01
  슈퍼마켓 및 잡화점            36  5626100.86    202539631    4828445    6411664  294360.59 2023-01-01 2025-12-01
         백화점            36  3401036.25    122437305    2967662    3998041  289825.16 2023-01-01 2025-12-01
        대형마트            36  3060581.47    110180933    2473948    3861641  262166.90 2023-01-01 2025-12-01
         편의점            36  2620157.42     94325667    2160538    2866979  185835.31 2023-01-01 2025-12-01
         면세점            36  1128645.25     40631229     797391    1590894  145459.83 2023-01-01 2025-12-01
channel_c1_nm     대형마트      면세점    무점

2026-02-11 21:26:21,362 - INFO - Excel 파일 저장 완료: kosis_retail_sales.xlsx
2026-02-11 21:26:21,363 - INFO - === 데이터 수집 완료 ===



=== YoY 성장률 피벗 테이블 전체 ===
channel_c1_nm   대형마트    면세점  무점포 소매   백화점  슈퍼마켓 및 잡화점  승용차 및 연료 소매점  전문소매점  \
date                                                                         
2024-01-01     -6.74  99.51   10.29  6.34       -4.39         -2.03  -6.77   
2024-02-01     26.36 -16.06   10.01  6.08       18.49        -10.51  -1.01   
2024-03-01     10.13  -2.88    5.37  6.40        3.72         -4.06  -4.87   
2024-04-01     -1.48   6.44   12.21 -4.60       -1.29         -1.53  -5.08   
2024-05-01      5.53   8.42    5.26 -5.09        0.81         -1.62  -4.73   
2024-06-01      2.82  12.02    4.32  1.93       -1.85         -7.76  -2.63   
2024-07-01     -5.79  15.86    5.50 -4.24       -2.37          4.02   0.69   
2024-08-01      3.69   6.95    0.79  0.86        1.52         -0.53  -2.94   
2024-09-01     -4.09 -10.04    0.94  0.46       -3.73         -2.60   0.62   
2024-10-01     -4.26 -16.40    0.81 -2.65        1.28         -0.37  -2.21   
2024-11-01      0.19 -12.16   -0.41  

In [17]:
 # 수집기 초기화
collector = KOSISRetailSalesCollector(kosis_key)
# 데이터 수집 실행 (최근 36개월)
results = collector.run_collection(months=100, save_excel=True)
processed_df = results['processed']
yoy_pivot_df = results['yoy_pivot']
# growth_data = collector.calculate_growth_rate(processed_df)

2026-02-11 21:26:24,956 - INFO - === KOSIS 판매채널별 판매액 데이터 수집 시작 ===
2026-02-11 21:26:24,957 - INFO - KOSIS 판매채널별 판매액 데이터 요청 (최근 100개월)
2026-02-11 21:26:25,294 - INFO - 원본 데이터 수집 완료: 576 rows
2026-02-11 21:26:25,295 - INFO - 컬럼: ['C1_OBJ_NM', 'DT', 'C1', 'PRD_SE', 'UNIT_NM_ENG', 'ITM_ID', 'TBL_ID', 'ITM_NM', 'TBL_NM', 'PRD_DE', 'LST_CHN_DE', 'C1_NM_ENG', 'C1_NM', 'UNIT_NM', 'ITM_NM_ENG', 'ORG_ID', 'C1_OBJ_NM_ENG']
2026-02-11 21:26:25,296 - INFO - 데이터 전처리 시작
2026-02-11 21:26:25,304 - INFO - 결측치 확인:
C1_OBJ_NM        0
DT               0
C1               0
PRD_SE           0
UNIT_NM_ENG      0
ITM_ID           0
TBL_ID           0
ITM_NM           0
TBL_NM           0
PRD_DE           0
LST_CHN_DE       0
C1_NM_ENG        0
C1_NM            0
UNIT_NM          0
ITM_NM_ENG       0
ORG_ID           0
C1_OBJ_NM_ENG    0
date             0
year             0
month            0
year_month       0
channel_c1_nm    0
channel_c1       0
sales_amount     0
unit             0
item_name        0
table

     channel  record_count   avg_sales  total_sales  min_sales  max_sales  std_sales first_date  last_date
       전문소매점            72 15244042.07   1097571029   12485052   17026074  938448.23 2020-01-01 2025-12-01
      무점포 소매            72 10546016.35    759313177    7373949   12709635 1187445.25 2020-01-01 2025-12-01
승용차 및 연료 소매점            72 10303943.64    741883942    6967752   12266219 1151976.35 2020-01-01 2025-12-01
  슈퍼마켓 및 잡화점            72  5508419.22    396606184    4476276    6411664  348982.09 2020-01-01 2025-12-01
         백화점            72  3073786.43    221312623    1698069    3998041  511787.57 2020-01-01 2025-12-01
        대형마트            72  2962551.24    213303689    2390074    3861641  277048.14 2020-01-01 2025-12-01
         편의점            72  2506480.01    180466561    1937346    2866979  242678.63 2020-01-01 2025-12-01
         면세점            72  1274818.64     91786942     797391    2024766  246926.00 2020-01-01 2025-12-01
channel_c1_nm     대형마트      면세점   무점포

2026-02-11 21:26:26,071 - INFO - Excel 파일 저장 완료: kosis_retail_sales.xlsx
2026-02-11 21:26:26,072 - INFO - === 데이터 수집 완료 ===


In [22]:
import requests
import pandas as pd
from datetime import datetime
import logging
from typing import Optional, Dict, List
import json
import os

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

class KOSISRetailCollector:
    """KOSIS 소매판매 데이터 수집 (판매채널별 + 제화별)"""

    def __init__(self, api_key: str, save_dir: str = None):
        self.api_key = api_key
        self.base_url = "https://kosis.kr/openapi/Param/statisticsParameterData.do"

        # 저장 디렉토리 설정
        if save_dir is None:
            self.save_dir = r'C:\Users\82108\OneDrive\바탕 화면\investment\data\analysis_results\KOSIS\retail_sales'
        else:
            self.save_dir = save_dir

        # 디렉토리가 없으면 생성
        os.makedirs(self.save_dir, exist_ok=True)
        logger.info(f"저장 디렉토리: {self.save_dir}")

    def get_data_by_table(self, table_id: str, months: int = 24) -> Optional[pd.DataFrame]:
        """
        KOSIS 통계표별 데이터 수집

        Parameters:
        -----------
        table_id : str
            통계표 ID (예: DT_1K41002, DT_1K41003)
        months : int
            최근 몇 개월 데이터 (기본값: 24개월)
        """
        params = {
            'method': 'getList',
            'apiKey': self.api_key,
            'itmId': 'T1',  # 판매액
            'objL1': 'ALL',  # 전체 업태
            'objL2': '',
            'objL3': '',
            'objL4': '',
            'objL5': '',
            'objL6': '',
            'objL7': '',
            'objL8': '',
            'format': 'json',
            'jsonVD': 'Y',
            'prdSe': 'M',  # 월별
            'newEstPrdCnt': str(months),
            'orgId': '101',
            'tblId': table_id
        }

        try:
            logger.info(f"KOSIS 데이터 요청 (테이블: {table_id}, 최근 {months}개월)")
            response = requests.get(self.base_url, params=params, timeout=30)
            response.raise_for_status()

            data = response.json()

            if not data:
                logger.warning(f"응답 데이터가 비어있습니다 (테이블: {table_id})")
                return None

            df = pd.DataFrame(data)
            logger.info(f"데이터 수집 완료 (테이블: {table_id}): {len(df)} rows")
            logger.info(f"컬럼: {df.columns.tolist()}")

            # 테이블 ID 추가
            df['source_table'] = table_id

            return df

        except requests.exceptions.RequestException as e:
            logger.error(f"API 요청 실패 (테이블: {table_id}): {e}")
            return None
        except Exception as e:
            logger.error(f"데이터 처리 중 오류 (테이블: {table_id}): {e}")
            return None

    def get_channel_sales(self, months: int = 24) -> Optional[pd.DataFrame]:
        """판매채널별 판매액 데이터 수집 (DT_1K41003)"""
        return self.get_data_by_table('DT_1K41003', months)

    def get_product_sales(self, months: int = 24) -> Optional[pd.DataFrame]:
        """제화별 판매액 데이터 수집 (DT_1K41002)"""
        return self.get_data_by_table('DT_1K41002', months)

    def process_data(self, df: pd.DataFrame, data_type: str = 'auto') -> pd.DataFrame:
        """
        데이터 전처리 및 정제

        Parameters:
        -----------
        data_type : str
            'channel' (판매채널별), 'product' (제화별), 'auto' (자동감지)
        """
        if df is None or df.empty:
            logger.warning("처리할 데이터가 없습니다")
            return pd.DataFrame()

        logger.info(f"데이터 전처리 시작 (타입: {data_type})")
        processed = df.copy()

        # 데이터 타입 자동 감지
        if data_type == 'auto':
            if 'source_table' in processed.columns:
                if processed['source_table'].iloc[0] == 'DT_1K41003':
                    data_type = 'channel'
                elif processed['source_table'].iloc[0] == 'DT_1K41002':
                    data_type = 'product'

        processed['data_type'] = data_type

        # 1. 날짜 처리
        if 'PRD_DE' in processed.columns:
            processed['date'] = pd.to_datetime(
                processed['PRD_DE'],
                format='%Y%m',
                errors='coerce'
            )
            processed['year'] = processed['date'].dt.year
            processed['month'] = processed['date'].dt.month
            processed['year_month'] = processed['PRD_DE']

        # 2. 카테고리 정보 (판매채널 또는 제화)
        category_cols = ['C1_NM', 'C2_NM', 'C1', 'C2']
        for col in category_cols:
            if col in processed.columns:
                processed[f'category_{col.lower()}'] = processed[col]

        # 3. 판매액 (숫자로 변환)
        if 'DT' in processed.columns:
            processed['sales_amount'] = pd.to_numeric(
                processed['DT'].astype(str).str.replace(',', '').str.strip(),
                errors='coerce'
            )

        # 4. 단위 정보
        if 'UNIT_NM' in processed.columns:
            processed['unit'] = processed['UNIT_NM']

        # 5. 항목명
        if 'ITM_NM' in processed.columns:
            processed['item_name'] = processed['ITM_NM']

        # 6. 통계표 정보
        if 'TBL_NM' in processed.columns:
            processed['table_name'] = processed['TBL_NM']

        if 'TBL_ID' in processed.columns:
            processed['table_id'] = processed['TBL_ID']

        if 'ORG_ID' in processed.columns:
            processed['org_id'] = processed['ORG_ID']

        # 7. 데이터 수집 시간
        processed['collected_at'] = datetime.now()

        # 8. 결측치 확인
        logger.info(f"결측치 확인:\n{processed.isnull().sum()}")

        return processed

    def create_summary(self, df: pd.DataFrame) -> pd.DataFrame:
        """카테고리별 요약 통계"""
        if df.empty or 'sales_amount' not in df.columns:
            logger.warning("요약 통계를 생성할 수 없습니다")
            return pd.DataFrame()

        # 카테고리 컬럼 찾기
        category_col = None
        for col in ['category_c1_nm', 'category_c2_nm', 'C1_NM', 'C2_NM']:
            if col in df.columns:
                category_col = col
                break

        if category_col is None:
            logger.warning("카테고리 정보를 찾을 수 없습니다")
            return pd.DataFrame()

        summary = df.groupby(category_col).agg({
            'sales_amount': ['count', 'mean', 'sum', 'min', 'max', 'std'],
            'date': ['min', 'max']
        }).round(2)

        summary.columns = ['_'.join(col).strip() for col in summary.columns.values]
        summary = summary.reset_index()
        summary.columns = [
            'category', 'record_count', 'avg_sales', 'total_sales',
            'min_sales', 'max_sales', 'std_sales', 'first_date', 'last_date'
        ]

        summary = summary.sort_values('total_sales', ascending=False)

        return summary

    def create_pivot_table(self, df: pd.DataFrame) -> pd.DataFrame:
        """날짜 x 카테고리 피벗 테이블 생성"""
        if df.empty:
            return pd.DataFrame()

        # 카테고리 컬럼 찾기
        category_col = None
        for col in ['category_c1_nm', 'category_c2_nm', 'C1_NM', 'C2_NM']:
            if col in df.columns:
                category_col = col
                break

        if category_col is None or 'date' not in df.columns:
            logger.warning("피벗 테이블을 생성할 수 없습니다")
            return pd.DataFrame()

        pivot = df.pivot_table(
            values='sales_amount',
            index='date',
            columns=category_col,
            aggfunc='sum'
        )

        pivot = pivot.sort_index()

        return pivot

    def calculate_growth_rate(self, df: pd.DataFrame) -> pd.DataFrame:
        """전년 동월 대비 성장률 계산"""
        if df.empty or 'date' not in df.columns:
            return pd.DataFrame()

        # 카테고리 컬럼 찾기
        category_col = None
        for col in ['category_c1_nm', 'category_c2_nm', 'C1_NM', 'C2_NM']:
            if col in df.columns:
                category_col = col
                break

        if category_col is None:
            return pd.DataFrame()

        df_sorted = df.sort_values(['date', category_col]).copy()

        # 전년 동월 대비 성장률
        df_sorted['yoy_growth'] = df_sorted.groupby(category_col)['sales_amount'].pct_change(12) * 100

        # 전월 대비 성장률
        df_sorted['mom_growth'] = df_sorted.groupby(category_col)['sales_amount'].pct_change(1) * 100

        return df_sorted

    def create_yoy_growth_pivot(self, df: pd.DataFrame) -> pd.DataFrame:
        """전년 동월 대비 성장률(YoY Growth) 피벗 테이블 생성"""
        if df.empty:
            logger.warning("데이터가 비어있습니다")
            return pd.DataFrame()

        if 'yoy_growth' not in df.columns:
            logger.warning("yoy_growth 컬럼이 없습니다. calculate_growth_rate()를 먼저 실행하세요")
            return pd.DataFrame()

        # 카테고리 컬럼 찾기
        category_col = None
        for col in ['category_c1_nm', 'category_c2_nm', 'C1_NM', 'C2_NM']:
            if col in df.columns:
                category_col = col
                break

        if category_col is None or 'date' not in df.columns:
            logger.warning("날짜 또는 카테고리 정보를 찾을 수 없습니다")
            return pd.DataFrame()

        # YoY 성장률 피벗 테이블 생성
        yoy_pivot = df.pivot_table(
            values='yoy_growth',
            index='date',
            columns=category_col,
            aggfunc='mean'
        )

        yoy_pivot = yoy_pivot.sort_index()
        yoy_pivot = yoy_pivot.round(2)

        logger.info(f"YoY 성장률 피벗 테이블 생성 완료: {yoy_pivot.shape}")

        return yoy_pivot

    def create_mom_growth_pivot(self, df: pd.DataFrame) -> pd.DataFrame:
        """전월 대비 성장률(MoM Growth) 피벗 테이블 생성"""
        if df.empty:
            logger.warning("데이터가 비어있습니다")
            return pd.DataFrame()

        if 'mom_growth' not in df.columns:
            logger.warning("mom_growth 컬럼이 없습니다. calculate_growth_rate()를 먼저 실행하세요")
            return pd.DataFrame()

        # 카테고리 컬럼 찾기
        category_col = None
        for col in ['category_c1_nm', 'category_c2_nm', 'C1_NM', 'C2_NM']:
            if col in df.columns:
                category_col = col
                break

        if category_col is None or 'date' not in df.columns:
            logger.warning("날짜 또는 카테고리 정보를 찾을 수 없습니다")
            return pd.DataFrame()

        # MoM 성장률 피벗 테이블 생성
        mom_pivot = df.pivot_table(
            values='mom_growth',
            index='date',
            columns=category_col,
            aggfunc='mean'
        )

        mom_pivot = mom_pivot.sort_index()
        mom_pivot = mom_pivot.round(2)

        logger.info(f"MoM 성장률 피벗 테이블 생성 완료: {mom_pivot.shape}")

        return mom_pivot

    def get_filename_with_date(self, base_name: str = 'kosis_retail_sales') -> str:
        """날짜가 포함된 파일명 생성"""
        today = datetime.now().strftime('%Y%m%d')
        filename = f"{base_name}_{today}.xlsx"
        filepath = os.path.join(self.save_dir, filename)
        return filepath

    def export_to_excel(self, results_dict: Dict[str, Dict[str, pd.DataFrame]],
                       filename: str = None):
        """여러 데이터셋을 하나의 Excel 파일로 저장"""

        # 파일명 생성 (날짜 포함)
        if filename is None:
            filepath = self.get_filename_with_date()
        else:
            filepath = os.path.join(self.save_dir, filename)

        try:
            with pd.ExcelWriter(filepath, engine='openpyxl') as writer:

                # 판매채널별 데이터
                if 'channel' in results_dict:
                    channel = results_dict['channel']

                    if not channel.get('processed', pd.DataFrame()).empty:
                        channel['processed'].to_excel(writer, sheet_name='채널_원본데이터', index=False)

                    if not channel.get('summary', pd.DataFrame()).empty:
                        channel['summary'].to_excel(writer, sheet_name='채널_요약', index=False)

                    if not channel.get('pivot', pd.DataFrame()).empty:
                        channel['pivot'].to_excel(writer, sheet_name='채널_판매액_피벗')

                    if not channel.get('growth', pd.DataFrame()).empty:
                        channel['growth'].to_excel(writer, sheet_name='채널_성장률분석', index=False)

                    if not channel.get('yoy_pivot', pd.DataFrame()).empty:
                        channel['yoy_pivot'].to_excel(writer, sheet_name='채널_YoY_피벗')

                    if not channel.get('mom_pivot', pd.DataFrame()).empty:
                        channel['mom_pivot'].to_excel(writer, sheet_name='채널_MoM_피벗')

                # 제화별 데이터
                if 'product' in results_dict:
                    product = results_dict['product']

                    if not product.get('processed', pd.DataFrame()).empty:
                        product['processed'].to_excel(writer, sheet_name='제화_원본데이터', index=False)

                    if not product.get('summary', pd.DataFrame()).empty:
                        product['summary'].to_excel(writer, sheet_name='제화_요약', index=False)

                    if not product.get('pivot', pd.DataFrame()).empty:
                        product['pivot'].to_excel(writer, sheet_name='제화_판매액_피벗')

                    if not product.get('growth', pd.DataFrame()).empty:
                        product['growth'].to_excel(writer, sheet_name='제화_성장률분석', index=False)

                    if not product.get('yoy_pivot', pd.DataFrame()).empty:
                        product['yoy_pivot'].to_excel(writer, sheet_name='제화_YoY_피벗')

                    if not product.get('mom_pivot', pd.DataFrame()).empty:
                        product['mom_pivot'].to_excel(writer, sheet_name='제화_MoM_피벗')

            logger.info(f"Excel 파일 저장 완료: {filepath}")

        except Exception as e:
            logger.error(f"Excel 저장 실패: {e}")

    def run_single_collection(self, table_id: str, data_type: str, months: int = 24) -> Dict[str, pd.DataFrame]:
        """단일 테이블 수집 프로세스"""
        logger.info(f"=== {data_type} 데이터 수집 시작 (테이블: {table_id}) ===")

        # 1. 데이터 수집
        raw_data = self.get_data_by_table(table_id, months)
        if raw_data is None or raw_data.empty:
            logger.error(f"{data_type} 데이터 수집 실패")
            return {
                'raw': pd.DataFrame(),
                'processed': pd.DataFrame(),
                'summary': pd.DataFrame(),
                'pivot': pd.DataFrame(),
                'growth': pd.DataFrame(),
                'yoy_pivot': pd.DataFrame(),
                'mom_pivot': pd.DataFrame()
            }

        # 2. 데이터 처리
        processed_data = self.process_data(raw_data, data_type)

        # 3. 요약 통계 생성
        summary = self.create_summary(processed_data)
        if not summary.empty:
            logger.info(f"\n=== {data_type} 요약 통계 ===")
            print(summary.to_string(index=False))

        # 4. 판매액 피벗 테이블 생성
        pivot = self.create_pivot_table(processed_data)
        if not pivot.empty:
            logger.info(f"\n=== {data_type} 판매액 피벗 테이블 ===")
            print(pivot.tail(10))

        # 5. 성장률 계산
        growth_data = self.calculate_growth_rate(processed_data)

        # 6. YoY 성장률 피벗 테이블 생성
        yoy_pivot = self.create_yoy_growth_pivot(growth_data)
        if not yoy_pivot.empty:
            logger.info(f"\n=== {data_type} YoY 성장률 피벗 테이블 ===")
            print(yoy_pivot.tail(12))

        # 7. MoM 성장률 피벗 테이블 생성
        mom_pivot = self.create_mom_growth_pivot(growth_data)

        logger.info(f"=== {data_type} 데이터 수집 완료 ===\n")

        return {
            'raw': raw_data,
            'processed': processed_data,
            'summary': summary,
            'pivot': pivot,
            'growth': growth_data,
            'yoy_pivot': yoy_pivot,
            'mom_pivot': mom_pivot
        }

    def run_all_collection(self, months: int = 24, save_excel: bool = True) -> Dict[str, Dict[str, pd.DataFrame]]:
        """
        판매채널별 + 제화별 데이터 모두 수집

        Returns:
        --------
        dict: 'channel'과 'product' 키를 가진 딕셔너리
        """
        logger.info("=== KOSIS 소매판매 전체 데이터 수집 시작 ===\n")

        results = {}

        # 1. 판매채널별 데이터 수집 (DT_1K41003)
        results['channel'] = self.run_single_collection('DT_1K41003', 'channel', months)

        # 2. 제화별 데이터 수집 (DT_1K41002)
        results['product'] = self.run_single_collection('DT_1K41002', 'product', months)

        # 3. Excel 저장
        if save_excel:
            self.export_to_excel(results)

        logger.info("=== KOSIS 소매판매 전체 데이터 수집 완료 ===")

        return results


def main():
    # API 키 설정
    kosis_key = "ZTFhMjg1MzhmNmFiYWJlYmY3ZWUxZDA0ZDI2ZTM0YWU="

    # 저장 경로 설정
    save_directory = r'C:\Users\82108\OneDrive\바탕 화면\investment\data\analysis_results\KOSIS\retail_sales'

    # 수집기 초기화
    collector = KOSISRetailCollector(kosis_key, save_dir=save_directory)

    # 전체 데이터 수집 (판매채널별 + 제화별)
    all_results = collector.run_all_collection(months=36, save_excel=True)

    # 판매채널별 결과
    channel_results = all_results['channel']
    print("\n=== 판매채널별 YoY 성장률 (최근 6개월) ===")
    if not channel_results['yoy_pivot'].empty:
        print(channel_results['yoy_pivot'].tail(6))

    # 제화별 결과
    product_results = all_results['product']
    print("\n=== 제화별 YoY 성장률 (최근 6개월) ===")
    if not product_results['yoy_pivot'].empty:
        print(product_results['yoy_pivot'].tail(6))

    print(f"\n파일 저장 위치: {save_directory}")
    print(f"파일명 형식: kosis_retail_sales_YYYYMMDD.xlsx")


if __name__ == "__main__":
    main()

2026-02-11 21:40:11,423 - INFO - 저장 디렉토리: C:\Users\82108\OneDrive\바탕 화면\investment\data\analysis_results\KOSIS\retail_sales
2026-02-11 21:40:11,424 - INFO - === KOSIS 소매판매 전체 데이터 수집 시작 ===

2026-02-11 21:40:11,424 - INFO - === channel 데이터 수집 시작 (테이블: DT_1K41003) ===
2026-02-11 21:40:11,425 - INFO - KOSIS 데이터 요청 (테이블: DT_1K41003, 최근 36개월)
2026-02-11 21:40:11,696 - INFO - 데이터 수집 완료 (테이블: DT_1K41003): 288 rows
2026-02-11 21:40:11,697 - INFO - 컬럼: ['C1_OBJ_NM', 'DT', 'C1', 'PRD_SE', 'UNIT_NM_ENG', 'ITM_ID', 'TBL_ID', 'ITM_NM', 'TBL_NM', 'PRD_DE', 'LST_CHN_DE', 'C1_NM_ENG', 'C1_NM', 'UNIT_NM', 'ITM_NM_ENG', 'ORG_ID', 'C1_OBJ_NM_ENG']
2026-02-11 21:40:11,699 - INFO - 데이터 전처리 시작 (타입: channel)
2026-02-11 21:40:11,706 - INFO - 결측치 확인:
C1_OBJ_NM         0
DT                0
C1                0
PRD_SE            0
UNIT_NM_ENG       0
ITM_ID            0
TBL_ID            0
ITM_NM            0
TBL_NM            0
PRD_DE            0
LST_CHN_DE        0
C1_NM_ENG         0
C1_NM             0
UNIT

    category  record_count   avg_sales  total_sales  min_sales  max_sales  std_sales first_date  last_date
       전문소매점            36 15691888.94    564908002   14491139   17026074  677373.14 2023-01-01 2025-12-01
      무점포 소매            36 11436106.17    411699822    9986914   12709635  603605.65 2023-01-01 2025-12-01
승용차 및 연료 소매점            36 10824028.75    389665035    9274173   12266219  684763.12 2023-01-01 2025-12-01
  슈퍼마켓 및 잡화점            36  5626100.86    202539631    4828445    6411664  294360.59 2023-01-01 2025-12-01
         백화점            36  3401036.25    122437305    2967662    3998041  289825.16 2023-01-01 2025-12-01
        대형마트            36  3060581.47    110180933    2473948    3861641  262166.90 2023-01-01 2025-12-01
         편의점            36  2620157.42     94325667    2160538    2866979  185835.31 2023-01-01 2025-12-01
         면세점            36  1128645.25     40631229     797391    1590894  145459.83 2023-01-01 2025-12-01
category_c1_nm     대형마트      면세점    무

2026-02-11 21:40:12,111 - INFO - 데이터 수집 완료 (테이블: DT_1K41002): 720 rows
2026-02-11 21:40:12,112 - INFO - 컬럼: ['C1_OBJ_NM', 'DT', 'C1', 'PRD_SE', 'UNIT_NM_ENG', 'ITM_ID', 'TBL_ID', 'ITM_NM', 'TBL_NM', 'PRD_DE', 'LST_CHN_DE', 'C1_NM_ENG', 'C1_NM', 'UNIT_NM', 'ITM_NM_ENG', 'ORG_ID', 'C1_OBJ_NM_ENG']
2026-02-11 21:40:12,114 - INFO - 데이터 전처리 시작 (타입: product)
2026-02-11 21:40:12,122 - INFO - 결측치 확인:
C1_OBJ_NM         0
DT                0
C1                0
PRD_SE            0
UNIT_NM_ENG       0
ITM_ID            0
TBL_ID            0
ITM_NM            0
TBL_NM            0
PRD_DE            0
LST_CHN_DE        0
C1_NM_ENG         0
C1_NM             0
UNIT_NM           0
ITM_NM_ENG        0
ORG_ID            0
C1_OBJ_NM_ENG     0
source_table      0
data_type         0
date              0
year              0
month             0
year_month        0
category_c1_nm    0
category_c1       0
sales_amount      0
unit              0
item_name         0
table_name        0
table_id          0
org_

  category  record_count   avg_sales  total_sales  min_sales  max_sales  std_sales first_date  last_date
        합계            36 53788545.11   1936387624   48625429   57687394 1987300.91 2023-01-01 2025-12-01
합계(승용차 제외)            36 48272009.64   1737792347   43423441   51436615 1708670.14 2023-01-01 2025-12-01
      비내구재            36 29855716.00   1074805776   26341190   32769820 1343425.02 2023-01-01 2025-12-01
      음식료품            36 15034936.03    541257697   12480241   17766442 1110545.40 2023-01-01 2025-12-01
       내구재            36 13425676.11    483324340   11590025   14858028  748771.77 2023-01-01 2025-12-01
      준내구재            36 10507153.00    378257508    8658700   12197737 1124360.67 2023-01-01 2025-12-01
        의복            36  5785716.64    208285799    4371774    7281203  852234.45 2023-01-01 2025-12-01
       승용차            36  5516535.47    198595277    4159036    6642089  614598.55 2023-01-01 2025-12-01
      차량연료            36  4903817.25    176537421    43

2026-02-11 21:40:13,484 - INFO - Excel 파일 저장 완료: C:\Users\82108\OneDrive\바탕 화면\investment\data\analysis_results\KOSIS\retail_sales\kosis_retail_sales_20260211.xlsx
2026-02-11 21:40:13,486 - INFO - === KOSIS 소매판매 전체 데이터 수집 완료 ===



=== 판매채널별 YoY 성장률 (최근 6개월) ===
category_c1_nm   대형마트    면세점  무점포 소매   백화점  슈퍼마켓 및 잡화점  승용차 및 연료 소매점  전문소매점  \
date                                                                          
2025-07-01       0.17 -19.86    6.55  1.10       -0.27          7.49   4.61   
2025-08-01     -10.66 -16.13    4.30  3.26       -5.70          5.33   3.77   
2025-09-01      -7.31 -10.60   14.84  3.49       -8.15         12.60   0.42   
2025-10-01      10.08  -5.66    1.30  8.20        4.30         -3.64   5.79   
2025-11-01      -4.91  -1.74    3.87  6.29       -1.09          6.52   3.87   
2025-12-01      -5.59 -11.00    4.82  3.14       -2.87          9.06   4.33   

category_c1_nm   편의점  
date                  
2025-07-01      3.03  
2025-08-01      0.22  
2025-09-01      0.15  
2025-10-01     -0.16  
2025-11-01     -0.13  
2025-12-01      0.98  

=== 제화별 YoY 성장률 (최근 6개월) ===
category_c1_nm    가구   가전제품  기타내구재  기타비내구재  기타준내구재    내구재  비내구재  서적 문구    승용차  \
date                                    

In [23]:
# API 키 설정
kosis_key = "ZTFhMjg1MzhmNmFiYWJlYmY3ZWUxZDA0ZDI2ZTM0YWU="

# 저장 경로 설정
save_directory = r'C:\Users\82108\OneDrive\바탕 화면\investment\data\analysis_results\KOSIS\retail_sales'

# 수집기 초기화
collector = KOSISRetailCollector(kosis_key, save_dir=save_directory)

# 전체 데이터 수집 (판매채널별 + 제화별)
all_results = collector.run_all_collection(months=36, save_excel=True)

# 판매채널별 결과
channel_results = all_results['channel']
print("\n=== 판매채널별 YoY 성장률 (최근 6개월) ===")
if not channel_results['yoy_pivot'].empty:
    print(channel_results['yoy_pivot'].tail(6))

# 제화별 결과
product_results = all_results['product']
print("\n=== 제화별 YoY 성장률 (최근 6개월) ===")
if not product_results['yoy_pivot'].empty:
    print(product_results['yoy_pivot'].tail(6))

print(f"\n파일 저장 위치: {save_directory}")
print(f"파일명 형식: kosis_retail_sales_YYYYMMDD.xlsx")

2026-02-11 21:42:32,251 - INFO - 저장 디렉토리: C:\Users\82108\OneDrive\바탕 화면\investment\data\analysis_results\KOSIS\retail_sales
2026-02-11 21:42:32,251 - INFO - === KOSIS 소매판매 전체 데이터 수집 시작 ===

2026-02-11 21:42:32,251 - INFO - === channel 데이터 수집 시작 (테이블: DT_1K41003) ===
2026-02-11 21:42:32,252 - INFO - KOSIS 데이터 요청 (테이블: DT_1K41003, 최근 36개월)
2026-02-11 21:42:32,511 - INFO - 데이터 수집 완료 (테이블: DT_1K41003): 288 rows
2026-02-11 21:42:32,512 - INFO - 컬럼: ['C1_OBJ_NM', 'DT', 'C1', 'PRD_SE', 'UNIT_NM_ENG', 'ITM_ID', 'TBL_ID', 'ITM_NM', 'TBL_NM', 'PRD_DE', 'LST_CHN_DE', 'C1_NM_ENG', 'C1_NM', 'UNIT_NM', 'ITM_NM_ENG', 'ORG_ID', 'C1_OBJ_NM_ENG']
2026-02-11 21:42:32,514 - INFO - 데이터 전처리 시작 (타입: channel)
2026-02-11 21:42:32,522 - INFO - 결측치 확인:
C1_OBJ_NM         0
DT                0
C1                0
PRD_SE            0
UNIT_NM_ENG       0
ITM_ID            0
TBL_ID            0
ITM_NM            0
TBL_NM            0
PRD_DE            0
LST_CHN_DE        0
C1_NM_ENG         0
C1_NM             0
UNIT

    category  record_count   avg_sales  total_sales  min_sales  max_sales  std_sales first_date  last_date
       전문소매점            36 15691888.94    564908002   14491139   17026074  677373.14 2023-01-01 2025-12-01
      무점포 소매            36 11436106.17    411699822    9986914   12709635  603605.65 2023-01-01 2025-12-01
승용차 및 연료 소매점            36 10824028.75    389665035    9274173   12266219  684763.12 2023-01-01 2025-12-01
  슈퍼마켓 및 잡화점            36  5626100.86    202539631    4828445    6411664  294360.59 2023-01-01 2025-12-01
         백화점            36  3401036.25    122437305    2967662    3998041  289825.16 2023-01-01 2025-12-01
        대형마트            36  3060581.47    110180933    2473948    3861641  262166.90 2023-01-01 2025-12-01
         편의점            36  2620157.42     94325667    2160538    2866979  185835.31 2023-01-01 2025-12-01
         면세점            36  1128645.25     40631229     797391    1590894  145459.83 2023-01-01 2025-12-01
category_c1_nm     대형마트      면세점    무

2026-02-11 21:42:32,839 - INFO - 데이터 수집 완료 (테이블: DT_1K41002): 720 rows
2026-02-11 21:42:32,840 - INFO - 컬럼: ['C1_OBJ_NM', 'DT', 'C1', 'PRD_SE', 'UNIT_NM_ENG', 'ITM_ID', 'TBL_ID', 'ITM_NM', 'TBL_NM', 'PRD_DE', 'LST_CHN_DE', 'C1_NM_ENG', 'C1_NM', 'UNIT_NM', 'ITM_NM_ENG', 'ORG_ID', 'C1_OBJ_NM_ENG']
2026-02-11 21:42:32,841 - INFO - 데이터 전처리 시작 (타입: product)
2026-02-11 21:42:32,849 - INFO - 결측치 확인:
C1_OBJ_NM         0
DT                0
C1                0
PRD_SE            0
UNIT_NM_ENG       0
ITM_ID            0
TBL_ID            0
ITM_NM            0
TBL_NM            0
PRD_DE            0
LST_CHN_DE        0
C1_NM_ENG         0
C1_NM             0
UNIT_NM           0
ITM_NM_ENG        0
ORG_ID            0
C1_OBJ_NM_ENG     0
source_table      0
data_type         0
date              0
year              0
month             0
year_month        0
category_c1_nm    0
category_c1       0
sales_amount      0
unit              0
item_name         0
table_name        0
table_id          0
org_

  category  record_count   avg_sales  total_sales  min_sales  max_sales  std_sales first_date  last_date
        합계            36 53788545.11   1936387624   48625429   57687394 1987300.91 2023-01-01 2025-12-01
합계(승용차 제외)            36 48272009.64   1737792347   43423441   51436615 1708670.14 2023-01-01 2025-12-01
      비내구재            36 29855716.00   1074805776   26341190   32769820 1343425.02 2023-01-01 2025-12-01
      음식료품            36 15034936.03    541257697   12480241   17766442 1110545.40 2023-01-01 2025-12-01
       내구재            36 13425676.11    483324340   11590025   14858028  748771.77 2023-01-01 2025-12-01
      준내구재            36 10507153.00    378257508    8658700   12197737 1124360.67 2023-01-01 2025-12-01
        의복            36  5785716.64    208285799    4371774    7281203  852234.45 2023-01-01 2025-12-01
       승용차            36  5516535.47    198595277    4159036    6642089  614598.55 2023-01-01 2025-12-01
      차량연료            36  4903817.25    176537421    43

2026-02-11 21:42:34,429 - INFO - Excel 파일 저장 완료: C:\Users\82108\OneDrive\바탕 화면\investment\data\analysis_results\KOSIS\retail_sales\kosis_retail_sales_20260211.xlsx
2026-02-11 21:42:34,430 - INFO - === KOSIS 소매판매 전체 데이터 수집 완료 ===



=== 판매채널별 YoY 성장률 (최근 6개월) ===
category_c1_nm   대형마트    면세점  무점포 소매   백화점  슈퍼마켓 및 잡화점  승용차 및 연료 소매점  전문소매점  \
date                                                                          
2025-07-01       0.17 -19.86    6.55  1.10       -0.27          7.49   4.61   
2025-08-01     -10.66 -16.13    4.30  3.26       -5.70          5.33   3.77   
2025-09-01      -7.31 -10.60   14.84  3.49       -8.15         12.60   0.42   
2025-10-01      10.08  -5.66    1.30  8.20        4.30         -3.64   5.79   
2025-11-01      -4.91  -1.74    3.87  6.29       -1.09          6.52   3.87   
2025-12-01      -5.59 -11.00    4.82  3.14       -2.87          9.06   4.33   

category_c1_nm   편의점  
date                  
2025-07-01      3.03  
2025-08-01      0.22  
2025-09-01      0.15  
2025-10-01     -0.16  
2025-11-01     -0.13  
2025-12-01      0.98  

=== 제화별 YoY 성장률 (최근 6개월) ===
category_c1_nm    가구   가전제품  기타내구재  기타비내구재  기타준내구재    내구재  비내구재  서적 문구    승용차  \
date                                    

In [25]:
channel_results['pivot']

category_c1_nm,대형마트,면세점,무점포 소매,백화점,슈퍼마켓 및 잡화점,승용차 및 연료 소매점,전문소매점,편의점
date,,,,,,,,
2023-01-01,3485426,797391,10630927,3062624,5840069,9767551,15907278,2315240
2023-02-01,2473948,1090319,9986914,2988967,4828445,10363664,14732634,2160538
2023-03-01,2851714,1221714,11016889,3357840,5336950,11500534,16270167,2527029
2023-04-01,2848227,1174846,10280498,3490298,5463118,10708960,16006925,2536429
2023-05-01,2929501,1156823,11223025,3743887,5724140,10885475,16625151,2682666
2023-06-01,2864918,1070833,10688426,3193720,5729623,11350451,15564048,2693016
2023-07-01,3199158,990857,10748058,3245692,5791070,9899621,15059682,2741053
2023-08-01,3237279,1136577,10842814,2967662,5891812,10492762,14958845,2788446
2023-09-01,3528200,1327364,10963703,3327346,6411664,10698097,16050076,2777479


In [26]:
channel_results['yoy_pivot']

category_c1_nm,대형마트,면세점,무점포 소매,백화점,슈퍼마켓 및 잡화점,승용차 및 연료 소매점,전문소매점,편의점
date,,,,,,,,
2024-01-01,-6.74,99.51,10.29,6.34,-4.39,-2.03,-6.77,3.49
2024-02-01,26.36,-16.06,10.01,6.08,18.49,-10.51,-1.01,7.10
2024-03-01,10.13,-2.88,5.37,6.40,3.72,-4.06,-4.87,0.94
2024-04-01,-1.48,6.44,12.21,-4.60,-1.29,-1.53,-5.08,3.90
2024-05-01,5.53,8.42,5.26,-5.09,0.81,-1.62,-4.73,2.97
2024-06-01,2.82,12.02,4.32,1.93,-1.85,-7.76,-2.63,1.97
2024-07-01,-5.79,15.86,5.50,-4.24,-2.37,4.02,0.69,1.01
2024-08-01,3.69,6.95,0.79,0.86,1.52,-0.53,-2.94,2.59
2024-09-01,-4.09,-10.04,0.94,0.46,-3.73,-2.60,0.62,0.63


In [29]:
product_results['yoy_pivot']

category_c1_nm,가구,가전제품,기타내구재,기타비내구재,기타준내구재,내구재,비내구재,서적 문구,승용차,신발 및 가방,오락 취미 경기용품,음식료품,의복,의약품,준내구재,차량연료,통신기기 및 컴퓨터,합계,합계(승용차 제외),화장품
date,,,,,,,,,,,,,,,,,,,,
2024-01-01,26.14,2.51,8.41,-1.94,4.82,9.70,-2.84,-0.64,2.83,-0.86,4.26,-7.30,0.62,6.33,1.72,-5.05,24.39,0.76,0.58,18.05
2024-02-01,11.13,-6.53,13.97,0.56,3.86,-8.14,9.65,-0.82,-15.17,1.08,-0.99,22.07,-1.57,8.49,-0.06,-4.78,-15.15,3.01,5.19,-7.19
2024-03-01,16.80,-4.39,8.36,0.58,-0.01,-4.24,2.03,1.25,-9.25,2.42,-8.93,4.62,2.10,3.76,0.25,1.16,-8.74,0.04,1.24,-7.40
2024-04-01,10.47,0.40,0.60,0.42,0.08,-1.97,2.19,-3.29,-4.27,-0.14,-6.07,3.27,-1.41,4.74,-1.61,1.52,-5.82,0.34,0.89,-0.31
2024-05-01,11.88,-7.47,3.99,5.97,0.96,-3.68,2.82,-0.24,-6.09,0.31,-6.56,2.85,-6.42,4.32,-4.25,3.50,-5.25,-0.32,0.35,-2.67
2024-06-01,13.31,-3.74,-0.92,3.00,2.51,-8.60,1.48,-2.57,-17.85,8.62,-6.16,0.20,2.56,3.41,2.11,6.24,-2.62,-1.15,1.21,-2.14
2024-07-01,6.64,-7.43,4.59,4.30,1.04,2.43,2.75,-3.54,0.50,2.36,-8.59,-0.09,-4.18,7.62,-2.88,7.49,17.50,1.58,1.70,5.58
2024-08-01,4.87,-9.20,3.70,0.71,-2.55,-6.65,3.05,-4.44,-1.81,6.86,-8.03,4.16,1.13,10.48,-0.30,1.08,-23.45,-0.05,0.14,-0.95
2024-09-01,8.40,-3.55,7.14,-0.30,-2.94,6.57,-4.15,-6.94,3.96,5.78,-3.77,-4.23,0.44,8.12,-0.17,-9.18,23.49,-1.02,-1.54,-8.33


In [30]:
import requests
import pandas as pd
from datetime import datetime
import logging
from typing import Optional, Dict, List
import json
import os

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

class KOSISRetailCollector:
    """KOSIS 소매판매 데이터 수집 (판매채널별 + 제화별 + 온라인)"""

    def __init__(self, api_key: str, save_dir: str = None):
        self.api_key = api_key
        self.base_url = "https://kosis.kr/openapi/Param/statisticsParameterData.do"

        # 저장 디렉토리 설정
        if save_dir is None:
            self.save_dir = r'C:\Users\82108\OneDrive\바탕 화면\investment\data\analysis_results\KOSIS\retail_sales'
        else:
            self.save_dir = save_dir

        # 디렉토리가 없으면 생성
        os.makedirs(self.save_dir, exist_ok=True)
        logger.info(f"저장 디렉토리: {self.save_dir}")

    def get_data_by_table(self, table_id: str, item_id: str = 'T1',
                         obj_l1: str = 'ALL', obj_l2: str = '',
                         months: int = 24) -> Optional[pd.DataFrame]:
        """
        KOSIS 통계표별 데이터 수집

        Parameters:
        -----------
        table_id : str
            통계표 ID (예: DT_1K41002, DT_1K41003, DT_1KE10041)
        item_id : str
            항목 ID (기본값: 'T1')
        obj_l1 : str
            객체 레벨1 (기본값: 'ALL')
        obj_l2 : str
            객체 레벨2 (기본값: '')
        months : int
            최근 몇 개월 데이터 (기본값: 24개월)
        """
        params = {
            'method': 'getList',
            'apiKey': self.api_key,
            'itmId': item_id,
            'objL1': obj_l1,
            'objL2': obj_l2,
            'objL3': '',
            'objL4': '',
            'objL5': '',
            'objL6': '',
            'objL7': '',
            'objL8': '',
            'format': 'json',
            'jsonVD': 'Y',
            'prdSe': 'M',  # 월별
            'newEstPrdCnt': str(months),
            'orgId': '101',
            'tblId': table_id
        }

        try:
            logger.info(f"KOSIS 데이터 요청 (테이블: {table_id}, 최근 {months}개월)")
            response = requests.get(self.base_url, params=params, timeout=30)
            response.raise_for_status()

            data = response.json()

            if not data:
                logger.warning(f"응답 데이터가 비어있습니다 (테이블: {table_id})")
                return None

            df = pd.DataFrame(data)
            logger.info(f"데이터 수집 완료 (테이블: {table_id}): {len(df)} rows")
            logger.info(f"컬럼: {df.columns.tolist()}")

            # 테이블 ID 추가
            df['source_table'] = table_id

            return df

        except requests.exceptions.RequestException as e:
            logger.error(f"API 요청 실패 (테이블: {table_id}): {e}")
            return None
        except Exception as e:
            logger.error(f"데이터 처리 중 오류 (테이블: {table_id}): {e}")
            return None

    def get_channel_sales(self, months: int = 24) -> Optional[pd.DataFrame]:
        """판매채널별 판매액 데이터 수집 (DT_1K41003)"""
        return self.get_data_by_table('DT_1K41003', item_id='T1', months=months)

    def get_product_sales(self, months: int = 24) -> Optional[pd.DataFrame]:
        """제화별 판매액 데이터 수집 (DT_1K41002)"""
        return self.get_data_by_table('DT_1K41002', item_id='T1', months=months)

    def get_online_sales(self, months: int = 24) -> Optional[pd.DataFrame]:
        """온라인 소매판매액 데이터 수집 (DT_1KE10041)"""
        return self.get_data_by_table('DT_1KE10041', item_id='T20', obj_l1='ALL', obj_l2='ALL', months=months)

    def process_data(self, df: pd.DataFrame, data_type: str = 'auto') -> pd.DataFrame:
        """
        데이터 전처리 및 정제

        Parameters:
        -----------
        data_type : str
            'channel' (판매채널별), 'product' (제화별), 'online' (온라인), 'auto' (자동감지)
        """
        if df is None or df.empty:
            logger.warning("처리할 데이터가 없습니다")
            return pd.DataFrame()

        logger.info(f"데이터 전처리 시작 (타입: {data_type})")
        processed = df.copy()

        # 데이터 타입 자동 감지
        if data_type == 'auto':
            if 'source_table' in processed.columns:
                if processed['source_table'].iloc[0] == 'DT_1K41003':
                    data_type = 'channel'
                elif processed['source_table'].iloc[0] == 'DT_1K41002':
                    data_type = 'product'
                elif processed['source_table'].iloc[0] == 'DT_1KE10041':
                    data_type = 'online'

        processed['data_type'] = data_type

        # 1. 날짜 처리
        if 'PRD_DE' in processed.columns:
            processed['date'] = pd.to_datetime(
                processed['PRD_DE'],
                format='%Y%m',
                errors='coerce'
            )
            processed['year'] = processed['date'].dt.year
            processed['month'] = processed['date'].dt.month
            processed['year_month'] = processed['PRD_DE']

        # 2. 카테고리 정보 (판매채널, 제화, 온라인 상품군)
        category_cols = ['C1_NM', 'C2_NM', 'C1', 'C2']
        for col in category_cols:
            if col in processed.columns:
                processed[f'category_{col.lower()}'] = processed[col]

        # 3. 판매액 (숫자로 변환)
        if 'DT' in processed.columns:
            processed['sales_amount'] = pd.to_numeric(
                processed['DT'].astype(str).str.replace(',', '').str.strip(),
                errors='coerce'
            )

        # 4. 단위 정보
        if 'UNIT_NM' in processed.columns:
            processed['unit'] = processed['UNIT_NM']

        # 5. 항목명
        if 'ITM_NM' in processed.columns:
            processed['item_name'] = processed['ITM_NM']

        # 6. 통계표 정보
        if 'TBL_NM' in processed.columns:
            processed['table_name'] = processed['TBL_NM']

        if 'TBL_ID' in processed.columns:
            processed['table_id'] = processed['TBL_ID']

        if 'ORG_ID' in processed.columns:
            processed['org_id'] = processed['ORG_ID']

        # 7. 데이터 수집 시간
        processed['collected_at'] = datetime.now()

        # 8. 결측치 확인
        logger.info(f"결측치 확인:\n{processed.isnull().sum()}")

        return processed

    def create_summary(self, df: pd.DataFrame) -> pd.DataFrame:
        """카테고리별 요약 통계"""
        if df.empty or 'sales_amount' not in df.columns:
            logger.warning("요약 통계를 생성할 수 없습니다")
            return pd.DataFrame()

        # 카테고리 컬럼 찾기
        category_col = None
        for col in ['category_c1_nm', 'category_c2_nm', 'C1_NM', 'C2_NM']:
            if col in df.columns:
                category_col = col
                break

        if category_col is None:
            logger.warning("카테고리 정보를 찾을 수 없습니다")
            return pd.DataFrame()

        summary = df.groupby(category_col).agg({
            'sales_amount': ['count', 'mean', 'sum', 'min', 'max', 'std'],
            'date': ['min', 'max']
        }).round(2)

        summary.columns = ['_'.join(col).strip() for col in summary.columns.values]
        summary = summary.reset_index()
        summary.columns = [
            'category', 'record_count', 'avg_sales', 'total_sales',
            'min_sales', 'max_sales', 'std_sales', 'first_date', 'last_date'
        ]

        summary = summary.sort_values('total_sales', ascending=False)

        return summary

    def create_pivot_table(self, df: pd.DataFrame) -> pd.DataFrame:
        """날짜 x 카테고리 피벗 테이블 생성"""
        if df.empty:
            return pd.DataFrame()

        # 카테고리 컬럼 찾기
        category_col = None
        for col in ['category_c1_nm', 'category_c2_nm', 'C1_NM', 'C2_NM']:
            if col in df.columns:
                category_col = col
                break

        if category_col is None or 'date' not in df.columns:
            logger.warning("피벗 테이블을 생성할 수 없습니다")
            return pd.DataFrame()

        pivot = df.pivot_table(
            values='sales_amount',
            index='date',
            columns=category_col,
            aggfunc='sum'
        )

        pivot = pivot.sort_index()

        return pivot

    def calculate_growth_rate(self, df: pd.DataFrame) -> pd.DataFrame:
        """전년 동월 대비 성장률 계산"""
        if df.empty or 'date' not in df.columns:
            return pd.DataFrame()

        # 카테고리 컬럼 찾기
        category_col = None
        for col in ['category_c1_nm', 'category_c2_nm', 'C1_NM', 'C2_NM']:
            if col in df.columns:
                category_col = col
                break

        if category_col is None:
            return pd.DataFrame()

        df_sorted = df.sort_values(['date', category_col]).copy()

        # 전년 동월 대비 성장률
        df_sorted['yoy_growth'] = df_sorted.groupby(category_col)['sales_amount'].pct_change(12) * 100

        # 전월 대비 성장률
        df_sorted['mom_growth'] = df_sorted.groupby(category_col)['sales_amount'].pct_change(1) * 100

        return df_sorted

    def create_yoy_growth_pivot(self, df: pd.DataFrame) -> pd.DataFrame:
        """전년 동월 대비 성장률(YoY Growth) 피벗 테이블 생성"""
        if df.empty:
            logger.warning("데이터가 비어있습니다")
            return pd.DataFrame()

        if 'yoy_growth' not in df.columns:
            logger.warning("yoy_growth 컬럼이 없습니다. calculate_growth_rate()를 먼저 실행하세요")
            return pd.DataFrame()

        # 카테고리 컬럼 찾기
        category_col = None
        for col in ['category_c1_nm', 'category_c2_nm', 'C1_NM', 'C2_NM']:
            if col in df.columns:
                category_col = col
                break

        if category_col is None or 'date' not in df.columns:
            logger.warning("날짜 또는 카테고리 정보를 찾을 수 없습니다")
            return pd.DataFrame()

        # YoY 성장률 피벗 테이블 생성
        yoy_pivot = df.pivot_table(
            values='yoy_growth',
            index='date',
            columns=category_col,
            aggfunc='mean'
        )

        yoy_pivot = yoy_pivot.sort_index()
        yoy_pivot = yoy_pivot.round(2)

        logger.info(f"YoY 성장률 피벗 테이블 생성 완료: {yoy_pivot.shape}")

        return yoy_pivot

    def create_mom_growth_pivot(self, df: pd.DataFrame) -> pd.DataFrame:
        """전월 대비 성장률(MoM Growth) 피벗 테이블 생성"""
        if df.empty:
            logger.warning("데이터가 비어있습니다")
            return pd.DataFrame()

        if 'mom_growth' not in df.columns:
            logger.warning("mom_growth 컬럼이 없습니다. calculate_growth_rate()를 먼저 실행하세요")
            return pd.DataFrame()

        # 카테고리 컬럼 찾기
        category_col = None
        for col in ['category_c1_nm', 'category_c2_nm', 'C1_NM', 'C2_NM']:
            if col in df.columns:
                category_col = col
                break

        if category_col is None or 'date' not in df.columns:
            logger.warning("날짜 또는 카테고리 정보를 찾을 수 없습니다")
            return pd.DataFrame()

        # MoM 성장률 피벗 테이블 생성
        mom_pivot = df.pivot_table(
            values='mom_growth',
            index='date',
            columns=category_col,
            aggfunc='mean'
        )

        mom_pivot = mom_pivot.sort_index()
        mom_pivot = mom_pivot.round(2)

        logger.info(f"MoM 성장률 피벗 테이블 생성 완료: {mom_pivot.shape}")

        return mom_pivot

    def get_filename_with_date(self, base_name: str = 'kosis_retail_sales') -> str:
        """날짜가 포함된 파일명 생성"""
        today = datetime.now().strftime('%Y%m%d')
        filename = f"{base_name}_{today}.xlsx"
        filepath = os.path.join(self.save_dir, filename)
        return filepath

    def export_to_excel(self, results_dict: Dict[str, Dict[str, pd.DataFrame]],
                       filename: str = None):
        """여러 데이터셋을 하나의 Excel 파일로 저장"""

        # 파일명 생성 (날짜 포함)
        if filename is None:
            filepath = self.get_filename_with_date()
        else:
            filepath = os.path.join(self.save_dir, filename)

        try:
            with pd.ExcelWriter(filepath, engine='openpyxl') as writer:

                # 판매채널별 데이터
                if 'channel' in results_dict:
                    channel = results_dict['channel']

                    if not channel.get('processed', pd.DataFrame()).empty:
                        channel['processed'].to_excel(writer, sheet_name='채널_원본데이터', index=False)

                    if not channel.get('summary', pd.DataFrame()).empty:
                        channel['summary'].to_excel(writer, sheet_name='채널_요약', index=False)

                    if not channel.get('pivot', pd.DataFrame()).empty:
                        channel['pivot'].to_excel(writer, sheet_name='채널_판매액_피벗')

                    if not channel.get('growth', pd.DataFrame()).empty:
                        channel['growth'].to_excel(writer, sheet_name='채널_성장률분석', index=False)

                    if not channel.get('yoy_pivot', pd.DataFrame()).empty:
                        channel['yoy_pivot'].to_excel(writer, sheet_name='채널_YoY_피벗')

                    if not channel.get('mom_pivot', pd.DataFrame()).empty:
                        channel['mom_pivot'].to_excel(writer, sheet_name='채널_MoM_피벗')

                # 제화별 데이터
                if 'product' in results_dict:
                    product = results_dict['product']

                    if not product.get('processed', pd.DataFrame()).empty:
                        product['processed'].to_excel(writer, sheet_name='제화_원본데이터', index=False)

                    if not product.get('summary', pd.DataFrame()).empty:
                        product['summary'].to_excel(writer, sheet_name='제화_요약', index=False)

                    if not product.get('pivot', pd.DataFrame()).empty:
                        product['pivot'].to_excel(writer, sheet_name='제화_판매액_피벗')

                    if not product.get('growth', pd.DataFrame()).empty:
                        product['growth'].to_excel(writer, sheet_name='제화_성장률분석', index=False)

                    if not product.get('yoy_pivot', pd.DataFrame()).empty:
                        product['yoy_pivot'].to_excel(writer, sheet_name='제화_YoY_피벗')

                    if not product.get('mom_pivot', pd.DataFrame()).empty:
                        product['mom_pivot'].to_excel(writer, sheet_name='제화_MoM_피벗')

                # 온라인 판매 데이터
                if 'online' in results_dict:
                    online = results_dict['online']

                    if not online.get('processed', pd.DataFrame()).empty:
                        online['processed'].to_excel(writer, sheet_name='온라인_원본데이터', index=False)

                    if not online.get('summary', pd.DataFrame()).empty:
                        online['summary'].to_excel(writer, sheet_name='온라인_요약', index=False)

                    if not online.get('pivot', pd.DataFrame()).empty:
                        online['pivot'].to_excel(writer, sheet_name='온라인_판매액_피벗')

                    if not online.get('growth', pd.DataFrame()).empty:
                        online['growth'].to_excel(writer, sheet_name='온라인_성장률분석', index=False)

                    if not online.get('yoy_pivot', pd.DataFrame()).empty:
                        online['yoy_pivot'].to_excel(writer, sheet_name='온라인_YoY_피벗')

                    if not online.get('mom_pivot', pd.DataFrame()).empty:
                        online['mom_pivot'].to_excel(writer, sheet_name='온라인_MoM_피벗')

            logger.info(f"Excel 파일 저장 완료: {filepath}")

        except Exception as e:
            logger.error(f"Excel 저장 실패: {e}")

    def run_single_collection(self, table_id: str, data_type: str,
                             item_id: str = 'T1', obj_l1: str = 'ALL',
                             obj_l2: str = '', months: int = 24) -> Dict[str, pd.DataFrame]:
        """단일 테이블 수집 프로세스"""
        logger.info(f"=== {data_type} 데이터 수집 시작 (테이블: {table_id}) ===")

        # 1. 데이터 수집
        raw_data = self.get_data_by_table(table_id, item_id, obj_l1, obj_l2, months)
        if raw_data is None or raw_data.empty:
            logger.error(f"{data_type} 데이터 수집 실패")
            return {
                'raw': pd.DataFrame(),
                'processed': pd.DataFrame(),
                'summary': pd.DataFrame(),
                'pivot': pd.DataFrame(),
                'growth': pd.DataFrame(),
                'yoy_pivot': pd.DataFrame(),
                'mom_pivot': pd.DataFrame()
            }

        # 2. 데이터 처리
        processed_data = self.process_data(raw_data, data_type)

        # 3. 요약 통계 생성
        summary = self.create_summary(processed_data)
        if not summary.empty:
            logger.info(f"\n=== {data_type} 요약 통계 ===")
            print(summary.to_string(index=False))

        # 4. 판매액 피벗 테이블 생성
        pivot = self.create_pivot_table(processed_data)
        if not pivot.empty:
            logger.info(f"\n=== {data_type} 판매액 피벗 테이블 ===")
            print(pivot.tail(10))

        # 5. 성장률 계산
        growth_data = self.calculate_growth_rate(processed_data)

        # 6. YoY 성장률 피벗 테이블 생성
        yoy_pivot = self.create_yoy_growth_pivot(growth_data)
        if not yoy_pivot.empty:
            logger.info(f"\n=== {data_type} YoY 성장률 피벗 테이블 ===")
            print(yoy_pivot.tail(12))

        # 7. MoM 성장률 피벗 테이블 생성
        mom_pivot = self.create_mom_growth_pivot(growth_data)

        logger.info(f"=== {data_type} 데이터 수집 완료 ===\n")

        return {
            'raw': raw_data,
            'processed': processed_data,
            'summary': summary,
            'pivot': pivot,
            'growth': growth_data,
            'yoy_pivot': yoy_pivot,
            'mom_pivot': mom_pivot
        }

    def run_all_collection(self, months: int = 24, save_excel: bool = True) -> Dict[str, Dict[str, pd.DataFrame]]:
        """
        판매채널별 + 제화별 + 온라인 데이터 모두 수집

        Returns:
        --------
        dict: 'channel', 'product', 'online' 키를 가진 딕셔너리
        """
        logger.info("=== KOSIS 소매판매 전체 데이터 수집 시작 ===\n")

        results = {}

        # 1. 판매채널별 데이터 수집 (DT_1K41003)
        results['channel'] = self.run_single_collection(
            table_id='DT_1K41003',
            data_type='channel',
            item_id='T1',
            obj_l1='ALL',
            obj_l2='',
            months=months
        )

        # 2. 제화별 데이터 수집 (DT_1K41002)
        results['product'] = self.run_single_collection(
            table_id='DT_1K41002',
            data_type='product',
            item_id='T1',
            obj_l1='ALL',
            obj_l2='',
            months=months
        )

        # 3. 온라인 소매판매 데이터 수집 (DT_1KE10041)
        results['online'] = self.run_single_collection(
            table_id='DT_1KE10041',
            data_type='online',
            item_id='T20',
            obj_l1='ALL',
            obj_l2='ALL',
            months=months
        )

        # 4. Excel 저장
        if save_excel:
            self.export_to_excel(results)

        logger.info("=== KOSIS 소매판매 전체 데이터 수집 완료 ===")

        return results


def main():
    # API 키 설정
    kosis_key = "ZTFhMjg1MzhmNmFiYWJlYmY3ZWUxZDA0ZDI2ZTM0YWU="

    # 저장 경로 설정
    save_directory = r'C:\Users\82108\OneDrive\바탕 화면\investment\data\analysis_results\KOSIS\retail_sales'

    # 수집기 초기화
    collector = KOSISRetailCollector(kosis_key, save_dir=save_directory)

    # 전체 데이터 수집 (판매채널별 + 제화별 + 온라인)
    all_results = collector.run_all_collection(months=36, save_excel=True)

    # 판매채널별 결과
    channel_results = all_results['channel']
    print("\n=== 판매채널별 YoY 성장률 (최근 6개월) ===")
    if not channel_results['yoy_pivot'].empty:
        print(channel_results['yoy_pivot'].tail(6))

    # 제화별 결과
    product_results = all_results['product']
    print("\n=== 제화별 YoY 성장률 (최근 6개월) ===")
    if not product_results['yoy_pivot'].empty:
        print(product_results['yoy_pivot'].tail(6))

    # 온라인 판매 결과
    online_results = all_results['online']
    print("\n=== 온라인 판매 YoY 성장률 (최근 6개월) ===")
    if not online_results['yoy_pivot'].empty:
        print(online_results['yoy_pivot'].tail(6))

    print(f"\n파일 저장 위치: {save_directory}")
    print(f"파일명 형식: kosis_retail_sales_YYYYMMDD.xlsx")


if __name__ == "__main__":
    main()

2026-02-11 21:54:02,898 - INFO - 저장 디렉토리: C:\Users\82108\OneDrive\바탕 화면\investment\data\analysis_results\KOSIS\retail_sales
2026-02-11 21:54:02,899 - INFO - === KOSIS 소매판매 전체 데이터 수집 시작 ===

2026-02-11 21:54:02,899 - INFO - === channel 데이터 수집 시작 (테이블: DT_1K41003) ===
2026-02-11 21:54:02,900 - INFO - KOSIS 데이터 요청 (테이블: DT_1K41003, 최근 36개월)
2026-02-11 21:54:03,127 - INFO - 데이터 수집 완료 (테이블: DT_1K41003): 288 rows
2026-02-11 21:54:03,127 - INFO - 컬럼: ['C1_OBJ_NM', 'DT', 'C1', 'PRD_SE', 'UNIT_NM_ENG', 'ITM_ID', 'TBL_ID', 'ITM_NM', 'TBL_NM', 'PRD_DE', 'LST_CHN_DE', 'C1_NM_ENG', 'C1_NM', 'UNIT_NM', 'ITM_NM_ENG', 'ORG_ID', 'C1_OBJ_NM_ENG']
2026-02-11 21:54:03,130 - INFO - 데이터 전처리 시작 (타입: channel)
2026-02-11 21:54:03,137 - INFO - 결측치 확인:
C1_OBJ_NM         0
DT                0
C1                0
PRD_SE            0
UNIT_NM_ENG       0
ITM_ID            0
TBL_ID            0
ITM_NM            0
TBL_NM            0
PRD_DE            0
LST_CHN_DE        0
C1_NM_ENG         0
C1_NM             0
UNIT

    category  record_count   avg_sales  total_sales  min_sales  max_sales  std_sales first_date  last_date
       전문소매점            36 15691888.94    564908002   14491139   17026074  677373.14 2023-01-01 2025-12-01
      무점포 소매            36 11436106.17    411699822    9986914   12709635  603605.65 2023-01-01 2025-12-01
승용차 및 연료 소매점            36 10824028.75    389665035    9274173   12266219  684763.12 2023-01-01 2025-12-01
  슈퍼마켓 및 잡화점            36  5626100.86    202539631    4828445    6411664  294360.59 2023-01-01 2025-12-01
         백화점            36  3401036.25    122437305    2967662    3998041  289825.16 2023-01-01 2025-12-01
        대형마트            36  3060581.47    110180933    2473948    3861641  262166.90 2023-01-01 2025-12-01
         편의점            36  2620157.42     94325667    2160538    2866979  185835.31 2023-01-01 2025-12-01
         면세점            36  1128645.25     40631229     797391    1590894  145459.83 2023-01-01 2025-12-01
category_c1_nm     대형마트      면세점    무

2026-02-11 21:54:03,538 - INFO - 데이터 수집 완료 (테이블: DT_1K41002): 720 rows
2026-02-11 21:54:03,538 - INFO - 컬럼: ['C1_OBJ_NM', 'DT', 'C1', 'PRD_SE', 'UNIT_NM_ENG', 'ITM_ID', 'TBL_ID', 'ITM_NM', 'TBL_NM', 'PRD_DE', 'LST_CHN_DE', 'C1_NM_ENG', 'C1_NM', 'UNIT_NM', 'ITM_NM_ENG', 'ORG_ID', 'C1_OBJ_NM_ENG']
2026-02-11 21:54:03,541 - INFO - 데이터 전처리 시작 (타입: product)
2026-02-11 21:54:03,549 - INFO - 결측치 확인:
C1_OBJ_NM         0
DT                0
C1                0
PRD_SE            0
UNIT_NM_ENG       0
ITM_ID            0
TBL_ID            0
ITM_NM            0
TBL_NM            0
PRD_DE            0
LST_CHN_DE        0
C1_NM_ENG         0
C1_NM             0
UNIT_NM           0
ITM_NM_ENG        0
ORG_ID            0
C1_OBJ_NM_ENG     0
source_table      0
data_type         0
date              0
year              0
month             0
year_month        0
category_c1_nm    0
category_c1       0
sales_amount      0
unit              0
item_name         0
table_name        0
table_id          0
org_

  category  record_count   avg_sales  total_sales  min_sales  max_sales  std_sales first_date  last_date
        합계            36 53788545.11   1936387624   48625429   57687394 1987300.91 2023-01-01 2025-12-01
합계(승용차 제외)            36 48272009.64   1737792347   43423441   51436615 1708670.14 2023-01-01 2025-12-01
      비내구재            36 29855716.00   1074805776   26341190   32769820 1343425.02 2023-01-01 2025-12-01
      음식료품            36 15034936.03    541257697   12480241   17766442 1110545.40 2023-01-01 2025-12-01
       내구재            36 13425676.11    483324340   11590025   14858028  748771.77 2023-01-01 2025-12-01
      준내구재            36 10507153.00    378257508    8658700   12197737 1124360.67 2023-01-01 2025-12-01
        의복            36  5785716.64    208285799    4371774    7281203  852234.45 2023-01-01 2025-12-01
       승용차            36  5516535.47    198595277    4159036    6642089  614598.55 2023-01-01 2025-12-01
      차량연료            36  4903817.25    176537421    43

2026-02-11 21:54:04,328 - INFO - 데이터 수집 완료 (테이블: DT_1KE10041): 2808 rows
2026-02-11 21:54:04,329 - INFO - 컬럼: ['C1_OBJ_NM', 'C2_NM', 'DT', 'C2', 'C1', 'PRD_SE', 'UNIT_NM_ENG', 'ITM_ID', 'TBL_ID', 'ITM_NM', 'TBL_NM', 'PRD_DE', 'LST_CHN_DE', 'C1_NM_ENG', 'C1_NM', 'UNIT_NM', 'ITM_NM_ENG', 'C2_OBJ_NM_ENG', 'C2_NM_ENG', 'ORG_ID', 'C1_OBJ_NM_ENG', 'C2_OBJ_NM']
2026-02-11 21:54:04,333 - INFO - 데이터 전처리 시작 (타입: online)
2026-02-11 21:54:04,350 - INFO - 결측치 확인:
C1_OBJ_NM         0
C2_NM             0
DT                0
C2                0
C1                0
PRD_SE            0
UNIT_NM_ENG       0
ITM_ID            0
TBL_ID            0
ITM_NM            0
TBL_NM            0
PRD_DE            0
LST_CHN_DE        0
C1_NM_ENG         0
C1_NM             0
UNIT_NM           0
ITM_NM_ENG        0
C2_OBJ_NM_ENG     0
C2_NM_ENG         0
ORG_ID            0
C1_OBJ_NM_ENG     0
C2_OBJ_NM         0
source_table      0
data_type         0
date              0
year              0
month             0
year_

   category  record_count   avg_sales  total_sales  min_sales  max_sales  std_sales first_date  last_date
         합계           108 14327378.65   1547356894    7266313   24290448 5328164.69 2023-01-01 2025-12-01
      음식서비스           108  2052768.11    221698956          0    3828145 1486377.63 2023-01-01 2025-12-01
      음·식료품           108  1899487.70    205144672     308023    3616725 1133701.06 2023-01-01 2025-12-01
 여행 및 교통서비스           108  1774250.13    191619014     122044    3051069 1149493.65 2023-01-01 2025-12-01
 가전·전자·통신기기           108  1240423.46    133965734     154612    2224553  714701.52 2023-01-01 2025-12-01
         의복           108  1221553.69    131927798     484114    2526798  520413.34 2023-01-01 2025-12-01
       생활용품           108  1043274.67    112673664      92001    1754207  675533.33 2023-01-01 2025-12-01
      가전·전자           108   892135.22     96350604      98244    1660920  539578.23 2023-01-01 2025-12-01
        화장품           108   715172.35     7723

2026-02-11 21:54:09,983 - INFO - Excel 파일 저장 완료: C:\Users\82108\OneDrive\바탕 화면\investment\data\analysis_results\KOSIS\retail_sales\kosis_retail_sales_20260211.xlsx
2026-02-11 21:54:09,984 - INFO - === KOSIS 소매판매 전체 데이터 수집 완료 ===



=== 판매채널별 YoY 성장률 (최근 6개월) ===
category_c1_nm   대형마트    면세점  무점포 소매   백화점  슈퍼마켓 및 잡화점  승용차 및 연료 소매점  전문소매점  \
date                                                                          
2025-07-01       0.17 -19.86    6.55  1.10       -0.27          7.49   4.61   
2025-08-01     -10.66 -16.13    4.30  3.26       -5.70          5.33   3.77   
2025-09-01      -7.31 -10.60   14.84  3.49       -8.15         12.60   0.42   
2025-10-01      10.08  -5.66    1.30  8.20        4.30         -3.64   5.79   
2025-11-01      -4.91  -1.74    3.87  6.29       -1.09          6.52   3.87   
2025-12-01      -5.59 -11.00    4.82  3.14       -2.87          9.06   4.33   

category_c1_nm   편의점  
date                  
2025-07-01      3.03  
2025-08-01      0.22  
2025-09-01      0.15  
2025-10-01     -0.16  
2025-11-01     -0.13  
2025-12-01      0.98  

=== 제화별 YoY 성장률 (최근 6개월) ===
category_c1_nm    가구   가전제품  기타내구재  기타비내구재  기타준내구재    내구재  비내구재  서적 문구    승용차  \
date                                    